# Código vFinal — 4 modelos (AHP-GPT e AHP-Claude × SAW e TOPSIS)

Estrutura:
1. Veto dos dados
2. Normalização dos dados
3. **AHP-GPT** → SAW-GPT → TOPSIS-GPT
4. **AHP-Claude** → SAW-Claude → TOPSIS-Claude
5. Comparação dos 4 modelos

In [1]:
#Importar as bibliotecas necessárias

import pandas as pd
import numpy as np
from google.colab import files

# Veto dos dados

In [2]:
# Upload da base consolidada
uploaded = files.upload()

# Identifica automaticamente o arquivo enviado
arquivo = next(iter(uploaded))

# Leitura da base
df = pd.read_excel(arquivo)

print(f"Arquivo carregado: {arquivo}")
print(f"Número de municípios: {len(df)}")

df.head()

Saving Base consolidada.xlsx to Base consolidada.xlsx
Arquivo carregado: Base consolidada.xlsx
Número de municípios: 645


,Codigo_municipio,Municipio,Fibra,Dist_230kV,FEC,Tarifa_A4,Dist_min_IX,ISH,IFA1001,T2M,Risco_Inund,Risco_Romp,Risco_Tornado,MDO_vinculos,MDO_custo,Roubo_furto_100k_hab
0,3500105,Adamantina,1.0,12.88,3.258380,0.4340,197.0,5,4.91,24.34,0,0,0,90,4370.23,589.6
1,3500204,Adolfo,1.0,49.48,5.273383,0.3243,62.6,4,4.36,24.38,0,0,0,0,NaN,563.0
2,3500303,Aguaí,1.0,41.09,3.764102,0.4358,92.3,3,4.40,21.67,0,0,0,16,5146.78,1429.3
3,3500402,Águas da Prata,1.0,15.18,2.720086,0.4358,113.4,4,4.91,21.67,2,0,0,2,6654.05,402.3
4,3500501,Águas de Lindóia,1.0,48.00,3.656270,0.3857,64.0,3,2.95,21.45,1,0,0,10,2086.43,429.5


In [3]:
# VETO 1 — CONECTIVIDADE / REDUNDÂNCIA
# São vetados municípios com Fibra <= 0,5.

df["Veto_conectividade"] = df["Fibra"] <= 0.5

In [4]:
# VETO 2 — RISCOS
# O município é vetado por riscos se atender a PELO MENOS uma das três condições abaixo.

df["Veto_tornado"] = df["Risco_Tornado"] >= 1

df["Veto_rompimento"] = df["Risco_Romp"] >= 1

df["Veto_inundacao"] = df["Risco_Inund"] >= 9


# Veto geral de riscos = qualquer uma das três condições
df["Veto_riscos"] = (
    df["Veto_tornado"]
    | df["Veto_rompimento"]
    | df["Veto_inundacao"]
)

In [5]:
# Veto final
df["Vetada"] = df["Veto_conectividade"] | df["Veto_riscos"]


# Classificação do motivo principal do veto
condicoes = [
    df["Veto_conectividade"] & df["Veto_riscos"],
    df["Veto_conectividade"],
    df["Veto_riscos"]
]

resultados = [
    "Ambos",
    "Conectividade / Redundância",
    "Riscos"
]

df["Motivo_veto"] = np.select(
    condicoes,
    resultados,
    default="Aprovada"
)

df[
    [
        "Codigo_municipio",
        "Municipio",
        "Vetada",
        "Motivo_veto"
    ]
].head()

,Codigo_municipio,Municipio,Vetada,Motivo_veto
0,3500105,Adamantina,False,Aprovada
1,3500204,Adolfo,False,Aprovada
2,3500303,Aguaí,False,Aprovada
3,3500402,Águas da Prata,False,Aprovada
4,3500501,Águas de Lindóia,False,Aprovada


In [6]:
# Municípios que seguem para normalização e ranking
df_elegiveis = df[df["Vetada"] == False].copy()

# Municípios eliminados pelos critérios de veto
df_excluidos = df[df["Vetada"] == True].copy()

print("Municípios iniciais:", len(df))
print("Municípios excluídos:", len(df_excluidos))
print("Municípios elegíveis:", len(df_elegiveis))

Municípios iniciais: 645
Municípios excluídos: 147
Municípios elegíveis: 498


In [7]:
#Identificação dos motivos de veto
resumo = (
    df["Motivo_veto"]
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="Numero_municipios")
)

resumo

,Status,Numero_municipios
0,Aprovada,498
1,Conectividade / Redundância,138
2,Riscos,9


# Normalização dos dados

In [8]:
# DEFINIÇÃO DOS CRITÉRIOS DE RANKING

# Critérios diretamente proporcionais:
# quanto maior o valor, melhor
criterios_diretos = [
    "ISH",
    "MDO_vinculos"
]

# Critérios inversamente proporcionais:
# quanto menor o valor, melhor
criterios_inversos = [
    "Dist_230kV",
    "FEC",
    "Tarifa_A4",
    "Dist_min_IX",
    "IFA1001",
    "T2M",
    "MDO_custo",
    "Roubo_furto_100k_hab"
]

criterios_ranking = criterios_diretos + criterios_inversos

print("Critérios diretos:", criterios_diretos)
print("Critérios inversos:", criterios_inversos)
print("Total de critérios de ranking:", len(criterios_ranking))

Critérios diretos: ['ISH', 'MDO_vinculos']
Critérios inversos: ['Dist_230kV', 'FEC', 'Tarifa_A4', 'Dist_min_IX', 'IFA1001', 'T2M', 'MDO_custo', 'Roubo_furto_100k_hab']
Total de critérios de ranking: 10


In [9]:
# DIAGNÓSTICO DE ZEROS E VALORES NULOs NOS CRITÉRIOS INVERSOS

diagnostico_inversos = pd.DataFrame({
    "Criterio": criterios_inversos,
    "Valores_nulos": [
        df_elegiveis[col].isna().sum()
        for col in criterios_inversos
    ],
    "Valores_zero": [
        (df_elegiveis[col] == 0).sum()
        for col in criterios_inversos
    ]
})

diagnostico_inversos

,Criterio,Valores_nulos,Valores_zero
0,Dist_230kV,0,0
1,FEC,0,0
2,Tarifa_A4,0,0
3,Dist_min_IX,0,4
4,IFA1001,21,0
5,T2M,0,0
6,MDO_custo,85,2
7,Roubo_furto_100k_hab,0,0


In [10]:
# Trabalhar em uma cópia para preservar df_elegiveis original
df_normalizacao = df_elegiveis.copy()

colunas_dados_indisponiveis = [
    "IFA1001",
    "MDO_custo"
]

#Conversão dos valores zeros em nulos na coluna MDO_Custo, já que são ausência de dados

for col in colunas_dados_indisponiveis:
    df_normalizacao.loc[
        df_normalizacao[col] == 0,
        col
    ] = np.nan

print("Valores nulos após tratamento:")

print(
    df_normalizacao[
        colunas_dados_indisponiveis
    ].isna().sum()
)

Valores nulos após tratamento:
IFA1001      21
MDO_custo    87
dtype: int64


In [11]:
# VALIDAÇÃO DOS CRITÉRIOS INVERSOS

for col in criterios_inversos:

    serie = pd.to_numeric(
        df_normalizacao[col],
        errors="coerce"
    )

    # IFA1001 e MDO_custo podem possuir NaN
    if col in colunas_dados_indisponiveis:
        continue

    # Dist_min_IX pode possuir zero legítimo
    if col == "Dist_min_IX":

        if serie.isna().any():
            raise ValueError(
                f"{col} possui valores nulos inesperados."
            )

        continue

    # Nos demais critérios não esperamos nulos nem zeros
    if serie.isna().any():
        raise ValueError(
            f"{col} possui valores nulos inesperados."
        )

    if (serie == 0).any():
        raise ValueError(
            f"{col} possui valores zero inesperados."
        )

print("Validação concluída.")

Validação concluída.


In [12]:
# TRANSFORMAÇÃO LOGARÍTMICA DE MDO_vinculos
# A variável é muito assimétrica (São Paulo tem 206.934 vínculos; a mediana é 12).
# O log1p comprime os valores extremos antes da normalização min-max.

df_normalizacao["MDO_vinculos"] = np.log1p(df_elegiveis["MDO_vinculos"])

display(df_normalizacao["MDO_vinculos"].describe().round(3))

,MDO_vinculos
count,498.000
mean,2.683
std,2.195
min,0.000
25%,0.693
50%,2.565
75%,4.147
max,12.240


In [13]:
# FUNÇÕES DE NORMALIZAÇÃO

#Direta

def normalizar_direto(serie):

    serie = pd.to_numeric(
        serie,
        errors="coerce"
    )

    if serie.isna().any():
        raise ValueError(
            f"Há valores nulos inesperados em {serie.name}."
        )

    maximo = serie.max()

    if maximo <= 0:
        raise ValueError(
            f"{serie.name} não possui máximo positivo."
        )

    return serie / maximo

#Inversa

def normalizar_inverso(
    serie,
    zero_real_melhor=False,
    nulo_score_zero=False
):

    serie = pd.to_numeric(
        serie,
        errors="coerce"
    )

    # O mínimo de referência considera somente valores estritamente positivos
    valores_positivos = serie[serie > 0]

    if valores_positivos.empty:
        raise ValueError(
            f"{serie.name} não possui valores positivos."
        )

    minimo_positivo = valores_positivos.min()

    # Criar série para armazenar os resultados
    normalizado = pd.Series(
        index=serie.index,
        dtype=float
    )

    # Valores positivos: mínimo positivo / valor observado
    mask_positivo = serie > 0

    normalizado.loc[mask_positivo] = (
        minimo_positivo
        / serie.loc[mask_positivo]
    )

    # Zero legítimo, como distância até IX.br
    if zero_real_melhor:

        normalizado.loc[
            serie == 0
        ] = 1.0

    elif (serie == 0).any():

        raise ValueError(
            f"{serie.name} possui zero sem regra definida."
        )

    # Valores ausentes por indisponibilidade de dados
    if nulo_score_zero:

        normalizado.loc[
            serie.isna()
        ] = 0.0

    elif serie.isna().any():

        raise ValueError(
            f"{serie.name} possui NaN sem regra definida."
        )

    return normalizado

In [14]:
# ============================================================
# FUNÇÕES DE NORMALIZAÇÃO MIN-MAX
# ============================================================

# Critérios diretamente proporcionais:
# maior valor = melhor desempenho
def normalizar_direto(serie):

    serie = pd.to_numeric(serie, errors="coerce")

    if serie.isna().any():
        raise ValueError(
            f"Há valores nulos ou não numéricos em {serie.name}."
        )

    minimo = serie.min()
    maximo = serie.max()

    if maximo == minimo:
        raise ValueError(
            f"{serie.name} possui todos os valores iguais."
        )

    return (serie - minimo) / (maximo - minimo)


# Critérios inversamente proporcionais:
# menor valor = melhor desempenho
def normalizar_inverso(serie, nulo_score_zero=False):

    serie = pd.to_numeric(serie, errors="coerce")

    # Para calcular mínimo e máximo,
    # desconsideramos valores ausentes
    valores_validos = serie.dropna()

    minimo = valores_validos.min()
    maximo = valores_validos.max()

    if maximo == minimo:
        raise ValueError(
            f"{serie.name} possui todos os valores válidos iguais."
        )

    normalizado = (
        (maximo - serie)
        / (maximo - minimo)
    )

    # Nos critérios em que NaN representa indisponibilidade,
    # atribuímos score normalizado igual a zero
    if nulo_score_zero:
        normalizado = normalizado.fillna(0)

    elif normalizado.isna().any():
        raise ValueError(
            f"{serie.name} possui valores nulos inesperados."
        )

    return normalizado


print("Funções de normalização Min-Max definidas.")

Funções de normalização Min-Max definidas.


## Matriz de desempenhos normalizados (comum aos 4 modelos)

A normalização não depende dos pesos, então essa matriz é calculada uma única vez
e usada pelos dois AHPs (GPT e Claude) no SAW e no TOPSIS.

In [15]:
# SAW - MATRIZ DE DESEMPENHOS NORMALIZADOS

# Criar matriz com os mesmos índices dos municípios elegíveis
matriz_normalizada = pd.DataFrame(
    index=df_normalizacao.index
)

# Aplicar normalização min-max aos critérios diretos
for col in criterios_diretos:
    matriz_normalizada[col] = normalizar_direto(
        df_normalizacao[col]
    )

# Aplicar normalização min-max aos critérios inversos
for col in criterios_inversos:
    matriz_normalizada[col] = normalizar_inverso(
        df_normalizacao[col],
        nulo_score_zero=(col in colunas_dados_indisponiveis)
    )

# Organizar colunas na ordem definida anteriormente
matriz_normalizada = matriz_normalizada[criterios_ranking]

print("Municípios:", len(matriz_normalizada))
print("Critérios:", len(matriz_normalizada.columns))

display(matriz_normalizada.head().round(4))

Municípios: 498
Critérios: 10


,ISH,MDO_vinculos,Dist_230kV,FEC,Tarifa_A4,Dist_min_IX,IFA1001,T2M,MDO_custo,Roubo_furto_100k_hab
0,1.0000,0.3685,0.8814,0.8562,0.1249,0.5169,0.8865,0.1504,0.8213,0.8355
1,0.6667,0.0000,0.5269,0.6728,0.8947,0.8465,0.8995,0.1444,0.0000,0.8434
2,0.3333,0.2315,0.6082,0.8102,0.1123,0.7737,0.8985,0.5519,0.7751,0.5851
3,0.6667,0.0898,0.8591,0.9052,0.1123,0.7219,0.8865,0.5519,0.6853,0.8914
4,0.3333,0.1959,0.5413,0.8200,0.4639,0.8431,0.9327,0.5850,0.9572,0.8833


# AHP — definição dos critérios (comum ao GPT e ao Claude)

In [16]:
#Importar bibliotecas

from itertools import combinations
from IPython.display import display

criterios_ahp = [
    "Dist_230kV",
    "FEC",
    "Tarifa_A4",
    "Dist_min_IX",
    "ISH",
    "IFA1001",
    "T2M",
    "MDO_custo",
    "MDO_vinculos",
    "Roubo_furto_100k_hab"
]

nomes_ahp = {
    "Dist_230kV": "Disponibilidade de energia",
    "FEC": "Confiabilidade de energia",
    "Tarifa_A4": "Custo de energia",
    "Dist_min_IX": "Latência",
    "ISH": "Disponibilidade de água",
    "IFA1001": "Custo da água",
    "T2M": "Refrigeração",
    "MDO_custo": "Custo da mão de obra",
    "MDO_vinculos": "Disponibilidade de mão de obra",
    "Roubo_furto_100k_hab": "Segurança física"
}

if len(criterios_ahp) != 10:
    raise ValueError("O AHP deve ter exatamente 10 critérios.")

if set(criterios_ahp) != set(criterios_ranking):
    raise ValueError("Os critérios não coincidem com os do ranking.")

print(f"Critérios AHP definidos: {len(criterios_ahp)}")

display(pd.DataFrame({
    "Coluna": criterios_ahp,
    "Critério": [nomes_ahp[c] for c in criterios_ahp]
}))

Critérios AHP definidos: 10


,Coluna,Critério
0,Dist_230kV,Disponibilidade de energia
1,FEC,Confiabilidade de energia
2,Tarifa_A4,Custo de energia
3,Dist_min_IX,Latência
4,ISH,Disponibilidade de água
5,IFA1001,Custo da água
6,T2M,Refrigeração
7,MDO_custo,Custo da mão de obra
8,MDO_vinculos,Disponibilidade de mão de obra
9,Roubo_furto_100k_hab,Segurança física


# AHP-GPT

In [17]:
# JULGAMENTOS GPT
# Exemplo: ("Dist_230kV", "FEC", "FEC", 3)
# Significa que FEC é 3 vezes mais importante.
# Nota 1 significa igual importância.

julgamentos_gpt = [
    ("Dist_230kV", "FEC", "Dist_230kV", 2),
    ("Dist_230kV", "Tarifa_A4", "Dist_230kV", 3),
    ("Dist_230kV", "Dist_min_IX", "Dist_230kV", 3),
    ("Dist_230kV", "ISH", "Dist_230kV", 3),
    ("Dist_230kV", "IFA1001", "Dist_230kV", 7),
    ("Dist_230kV", "T2M", "Dist_230kV", 3),
    ("Dist_230kV", "MDO_custo", "Dist_230kV", 7),
    ("Dist_230kV", "MDO_vinculos", "Dist_230kV", 5),
    ("Dist_230kV", "Roubo_furto_100k_hab", "Dist_230kV", 5),

    ("FEC", "Tarifa_A4", "FEC", 2),
    ("FEC", "Dist_min_IX", "FEC", 3),
    ("FEC", "ISH", "FEC", 3),
    ("FEC", "IFA1001", "FEC", 7),
    ("FEC", "T2M", "FEC", 3),
    ("FEC", "MDO_custo", "FEC", 7),
    ("FEC", "MDO_vinculos", "FEC", 5),
    ("FEC", "Roubo_furto_100k_hab", "FEC", 5),

    ("Tarifa_A4", "Dist_min_IX", "Tarifa_A4", 2),
    ("Tarifa_A4", "ISH", "Tarifa_A4", 2),
    ("Tarifa_A4", "IFA1001", "Tarifa_A4", 4),
    ("Tarifa_A4", "T2M", "Tarifa_A4", 1),
    ("Tarifa_A4", "MDO_custo", "Tarifa_A4", 4),
    ("Tarifa_A4", "MDO_vinculos", "Tarifa_A4", 3),
    ("Tarifa_A4", "Roubo_furto_100k_hab", "Tarifa_A4", 3),

    ("Dist_min_IX", "ISH", "Dist_min_IX", 1),
    ("Dist_min_IX", "IFA1001", "Dist_min_IX", 4),
    ("Dist_min_IX", "T2M", "Dist_min_IX", 1),
    ("Dist_min_IX", "MDO_custo", "Dist_min_IX", 4),
    ("Dist_min_IX", "MDO_vinculos", "Dist_min_IX", 2),
    ("Dist_min_IX", "Roubo_furto_100k_hab", "Dist_min_IX", 2),

    ("ISH", "IFA1001", "ISH", 4),
    ("ISH", "T2M", "ISH", 1),
    ("ISH", "MDO_custo", "ISH", 4),
    ("ISH", "MDO_vinculos", "ISH", 3),
    ("ISH", "Roubo_furto_100k_hab", "ISH", 3),

    ("IFA1001", "T2M", "T2M", 4),
    ("IFA1001", "MDO_custo", "IFA1001", 1),
    ("IFA1001", "MDO_vinculos", "MDO_vinculos", 2),
    ("IFA1001", "Roubo_furto_100k_hab", "Roubo_furto_100k_hab", 2),

    ("T2M", "MDO_custo", "T2M", 4),
    ("T2M", "MDO_vinculos", "T2M", 3),
    ("T2M", "Roubo_furto_100k_hab", "T2M", 3),

    ("MDO_custo", "MDO_vinculos", "MDO_vinculos", 2),
    ("MDO_custo", "Roubo_furto_100k_hab", "Roubo_furto_100k_hab", 2),

    ("MDO_vinculos", "Roubo_furto_100k_hab", "MDO_vinculos", 1)
]

print(f"Julgamentos GPT informados: {len(julgamentos_gpt)}")  # deve imprimir 45

Julgamentos GPT informados: 45


In [18]:
n = len(criterios_ahp)

# Criar a matriz com diagonal igual a 1
matriz_gpt = pd.DataFrame(
    np.eye(n, dtype=float),
    index=criterios_ahp,
    columns=criterios_ahp
)

# Preencher os julgamentos e seus valores recíprocos
for a, b, mais_importante, nota in julgamentos_gpt:

    if mais_importante == a:
        valor_ab = float(nota)
    else:
        valor_ab = 1.0 / nota

    matriz_gpt.loc[a, b] = valor_ab
    matriz_gpt.loc[b, a] = 1.0 / valor_ab

# Validar diagonal e reciprocidade
assert np.allclose(np.diag(matriz_gpt), 1.0)

assert np.allclose(
    matriz_gpt.to_numpy() * matriz_gpt.to_numpy().T,
    1.0
)

print(f"Matriz AHP-GPT criada: {n} x {n}")
display(matriz_gpt.round(3))

Matriz AHP-GPT criada: 10 x 10


,Dist_230kV,FEC,Tarifa_A4,Dist_min_IX,ISH,IFA1001,T2M,MDO_custo,MDO_vinculos,Roubo_furto_100k_hab
Dist_230kV,1.000,2.000,3.000,3.00,3.000,7.0,3.000,7.0,5.0,5.0
FEC,0.500,1.000,2.000,3.00,3.000,7.0,3.000,7.0,5.0,5.0
Tarifa_A4,0.333,0.500,1.000,2.00,2.000,4.0,1.000,4.0,3.0,3.0
Dist_min_IX,0.333,0.333,0.500,1.00,1.000,4.0,1.000,4.0,2.0,2.0
ISH,0.333,0.333,0.500,1.00,1.000,4.0,1.000,4.0,3.0,3.0
IFA1001,0.143,0.143,0.250,0.25,0.250,1.0,0.250,1.0,0.5,0.5
T2M,0.333,0.333,1.000,1.00,1.000,4.0,1.000,4.0,3.0,3.0
MDO_custo,0.143,0.143,0.250,0.25,0.250,1.0,0.250,1.0,0.5,0.5
MDO_vinculos,0.200,0.200,0.333,0.50,0.333,2.0,0.333,2.0,1.0,1.0
Roubo_furto_100k_hab,0.200,0.200,0.333,0.50,0.333,2.0,0.333,2.0,1.0,1.0


In [19]:
# NORMALIZAÇÃO DA MATRIZ AHP-GPT
# Dividir cada elemento pela soma de sua coluna

matriz_gpt_normalizada = matriz_gpt.div(
    matriz_gpt.sum(axis=0),
    axis=1
)

# CÁLCULO DOS PESOS
# Média de cada linha da matriz normalizada

pesos_gpt = matriz_gpt_normalizada.mean(axis=1)
pesos_gpt.name = "Peso_GPT"

# Tabela equivalente à Tabela 3.3 do TCC

tabela_pesos_gpt = pd.DataFrame({
    "Criterio": [nomes_ahp[col] for col in criterios_ahp],
    "Peso": pesos_gpt.loc[criterios_ahp].to_numpy()
})

tabela_pesos_gpt.loc[len(tabela_pesos_gpt)] = [
    "Total",
    tabela_pesos_gpt["Peso"].sum()
]

print("Matriz AHP-GPT normalizada:")
display(matriz_gpt_normalizada.round(4))

print("Pesos dos critérios (GPT):")
display(tabela_pesos_gpt.round(5))

Matriz AHP-GPT normalizada:


,Dist_230kV,FEC,Tarifa_A4,Dist_min_IX,ISH,IFA1001,T2M,MDO_custo,MDO_vinculos,Roubo_furto_100k_hab
Dist_230kV,0.2842,0.3857,0.3273,0.24,0.2466,0.1944,0.2687,0.1944,0.2083,0.2083
FEC,0.1421,0.1928,0.2182,0.24,0.2466,0.1944,0.2687,0.1944,0.2083,0.2083
Tarifa_A4,0.0947,0.0964,0.1091,0.16,0.1644,0.1111,0.0896,0.1111,0.1250,0.1250
Dist_min_IX,0.0947,0.0643,0.0545,0.08,0.0822,0.1111,0.0896,0.1111,0.0833,0.0833
ISH,0.0947,0.0643,0.0545,0.08,0.0822,0.1111,0.0896,0.1111,0.1250,0.1250
IFA1001,0.0406,0.0275,0.0273,0.02,0.0205,0.0278,0.0224,0.0278,0.0208,0.0208
T2M,0.0947,0.0643,0.1091,0.08,0.0822,0.1111,0.0896,0.1111,0.1250,0.1250
MDO_custo,0.0406,0.0275,0.0273,0.02,0.0205,0.0278,0.0224,0.0278,0.0208,0.0208
MDO_vinculos,0.0568,0.0386,0.0364,0.04,0.0274,0.0556,0.0299,0.0556,0.0417,0.0417
Roubo_furto_100k_hab,0.0568,0.0386,0.0364,0.04,0.0274,0.0556,0.0299,0.0556,0.0417,0.0417


Pesos dos critérios (GPT):


,Criterio,Peso
0,Disponibilidade de energia,0.25579
1,Confiabilidade de energia,0.21139
2,Custo de energia,0.11864
3,Latência,0.08542
4,Disponibilidade de água,0.09375
5,Custo da água,0.02556
6,Refrigeração,0.09921
7,Custo da mão de obra,0.02556
8,Disponibilidade de mão de obra,0.04235
9,Segurança física,0.04235


In [20]:
# Organizar os pesos na ordem de criterios_ranking

pesos_para_ranking_gpt = pesos_gpt.reindex(criterios_ranking)

# Validar a correspondência dos critérios
if pesos_para_ranking_gpt.isna().any():
    raise ValueError("Existem critérios sem pesos AHP-GPT.")

# Validar a soma dos pesos
assert np.isclose(pesos_para_ranking_gpt.sum(), 1.0)

print(f"Total de pesos: {len(pesos_para_ranking_gpt)}")
print(f"Soma dos pesos: {pesos_para_ranking_gpt.sum():.6f}")

print("Vetor de pesos GPT para SAW e TOPSIS:")
display(pesos_para_ranking_gpt.round(5).to_frame("Peso_GPT"))

Total de pesos: 10
Soma dos pesos: 1.000000
Vetor de pesos GPT para SAW e TOPSIS:


,Peso_GPT
ISH,0.09375
MDO_vinculos,0.04235
Dist_230kV,0.25579
FEC,0.21139
Tarifa_A4,0.11864
Dist_min_IX,0.08542
IFA1001,0.02556
T2M,0.09921
MDO_custo,0.02556
Roubo_furto_100k_hab,0.04235


# SAW — AHP-GPT

In [21]:
# SAW - CÁLCULO DOS SCORES (AHP-GPT)

# Organizar pesos AHP-GPT na ordem da matriz normalizada
pesos_saw_gpt = pesos_para_ranking_gpt.reindex(
    matriz_normalizada.columns
)

# Multiplicar cada critério normalizado pelo seu peso AHP-GPT
matriz_ponderada_saw_gpt = matriz_normalizada.mul(
    pesos_saw_gpt,
    axis=1
)

# Aplicar a Equação 3.5: soma dos valores ponderados
scores_saw_gpt = matriz_ponderada_saw_gpt.sum(axis=1)

print("Scores SAW-GPT calculados:", len(scores_saw_gpt))

display(
    scores_saw_gpt.head().round(4).to_frame("Score_SAW_GPT")
)

Scores SAW-GPT calculados: 498


,Score_SAW_GPT
0,0.6687
1,0.5910
2,0.5696
3,0.6851
4,0.6221


In [22]:
# SAW - RANKING DOS MUNICÍPIOS (AHP-GPT)

# Criar base de resultados com identificação dos municípios
ranking_saw_gpt = df_normalizacao[
    ["Codigo_municipio", "Municipio"]
].copy()

# Adicionar o score obtido pelo SAW
ranking_saw_gpt["Score_SAW_GPT"] = scores_saw_gpt

# Ordenar municípios pelo score, do maior para o menor
ranking_saw_gpt = ranking_saw_gpt.sort_values(
    by=["Score_SAW_GPT", "Municipio"],
    ascending=[False, True]
).reset_index(drop=True)

# Criar coluna de posição no ranking
ranking_saw_gpt.insert(
    0,
    "Ranking_SAW_GPT",
    ranking_saw_gpt.index + 1
)

# Apresentar os dez primeiros municípios
print("Ranking AHP-GPT-SAW - Top 10 municípios")

display(
    ranking_saw_gpt[
        ["Ranking_SAW_GPT", "Municipio", "Score_SAW_GPT"]
    ].head(10).round(4)
)

Ranking AHP-GPT-SAW - Top 10 municípios


,Ranking_SAW_GPT,Municipio,Score_SAW_GPT
0,1,São Caetano do Sul,0.8462
1,2,Piraju,0.7958
2,3,Diadema,0.7909
3,4,Osasco,0.7848
4,5,Sumaré,0.7806
5,6,Campinas,0.7786
6,7,Santo André,0.7733
7,8,Avaré,0.7728
8,9,Pedreira,0.7728
9,10,Sarutaiá,0.7702


# TOPSIS — AHP-GPT

In [23]:
# TOPSIS - MATRIZ PONDERADA (AHP-GPT)


# Organizar os pesos AHP-GPT na ordem da matriz normalizada
pesos_topsis_gpt = pesos_gpt.reindex(
    matriz_normalizada.columns
)


# Multiplicar cada critério normalizado pelo respectivo peso
# Equação 3.6
matriz_ponderada_topsis_gpt = matriz_normalizada.mul(
    pesos_topsis_gpt,
    axis=1
)


print("Municípios:", len(matriz_ponderada_topsis_gpt))
print("Critérios:", len(matriz_ponderada_topsis_gpt.columns))


display(matriz_ponderada_topsis_gpt.head().round(4))

Municípios: 498
Critérios: 10


,ISH,MDO_vinculos,Dist_230kV,FEC,Tarifa_A4,Dist_min_IX,IFA1001,T2M,MDO_custo,Roubo_furto_100k_hab
0,0.0938,0.0156,0.2254,0.1810,0.0148,0.0442,0.0227,0.0149,0.0210,0.0354
1,0.0625,0.0000,0.1348,0.1422,0.1062,0.0723,0.0230,0.0143,0.0000,0.0357
2,0.0313,0.0098,0.1556,0.1713,0.0133,0.0661,0.0230,0.0547,0.0198,0.0248
3,0.0625,0.0038,0.2197,0.1914,0.0133,0.0617,0.0227,0.0547,0.0175,0.0377
4,0.0313,0.0083,0.1384,0.1733,0.0550,0.0720,0.0238,0.0580,0.0245,0.0374


In [24]:
# TOPSIS - SOLUÇÕES IDEAIS (AHP-GPT)

# Solução Ideal Positiva - Equação 3.7
ideal_positivo_gpt = matriz_ponderada_topsis_gpt.max(axis=0)


# Solução Ideal Negativa - Equação 3.8
ideal_negativo_gpt = matriz_ponderada_topsis_gpt.min(axis=0)


# Organizar os resultados
tabela_ideais_gpt = pd.DataFrame({
    "Ideal_Positivo": ideal_positivo_gpt,
    "Ideal_Negativo": ideal_negativo_gpt
})


print("Soluções ideais por critério (GPT):")
display(tabela_ideais_gpt.round(4))

Soluções ideais por critério (GPT):


,Ideal_Positivo,Ideal_Negativo
ISH,0.0938,0.0
MDO_vinculos,0.0423,0.0
Dist_230kV,0.2558,0.0
FEC,0.2114,0.0
Tarifa_A4,0.1186,0.0
Dist_min_IX,0.0854,0.0
IFA1001,0.0256,0.0
T2M,0.0992,0.0
MDO_custo,0.0256,0.0
Roubo_furto_100k_hab,0.0423,0.0


In [25]:
# TOPSIS - DISTÂNCIAS EUCLIDIANAS (AHP-GPT)


# Distância até a Solução Ideal Positiva - Equação 3.9
distancia_positiva_gpt = np.sqrt(
    (
        matriz_ponderada_topsis_gpt.sub(
            ideal_positivo_gpt, axis=1
        ) ** 2
    ).sum(axis=1)
)


# Distância até a Solução Ideal Negativa - Equação 3.10
distancia_negativa_gpt = np.sqrt(
    (
        matriz_ponderada_topsis_gpt.sub(
            ideal_negativo_gpt, axis=1
        ) ** 2
    ).sum(axis=1)
)


# Consolidar as distâncias
tabela_distancias_gpt = pd.DataFrame({
    "Distancia_Positiva": distancia_positiva_gpt,
    "Distancia_Negativa": distancia_negativa_gpt
})


print("Distâncias calculadas (GPT):", len(tabela_distancias_gpt))


display(tabela_distancias_gpt.head().round(4))

Distâncias calculadas (GPT): 498


,Distancia_Positiva,Distancia_Negativa
0,0.1491,0.3118
1,0.1745,0.2466
2,0.1744,0.2523
3,0.1338,0.3131
4,0.1620,0.2538


In [26]:
# TOPSIS - SCORES E RANKING FINAL (AHP-GPT)


# Calcular o coeficiente de proximidade - Equação 3.11
scores_topsis_gpt = (
    distancia_negativa_gpt
    / (distancia_negativa_gpt + distancia_positiva_gpt)
)


# Criar a base de resultados
ranking_topsis_gpt = df_normalizacao[
    ["Codigo_municipio", "Municipio"]
].copy()


# Adicionar o score TOPSIS
ranking_topsis_gpt["Score_TOPSIS_GPT"] = scores_topsis_gpt


# Ordenar do maior para o menor score
ranking_topsis_gpt = ranking_topsis_gpt.sort_values(
    by=["Score_TOPSIS_GPT", "Municipio"],
    ascending=[False, True]
).reset_index(drop=True)


# Criar a posição no ranking
ranking_topsis_gpt.insert(
    0,
    "Ranking_TOPSIS_GPT",
    ranking_topsis_gpt.index + 1
)


# Apresentar os dez melhores municípios
print("Ranking AHP-GPT-TOPSIS - Top 10 municípios")


display(
    ranking_topsis_gpt[
        ["Ranking_TOPSIS_GPT", "Municipio", "Score_TOPSIS_GPT"]
    ].head(10).round(4)
)

Ranking AHP-GPT-TOPSIS - Top 10 municípios


,Ranking_TOPSIS_GPT,Municipio,Score_TOPSIS_GPT
0,1,São Caetano do Sul,0.8166
1,2,Piraju,0.7993
2,3,Avaré,0.7961
3,4,Diadema,0.7899
4,5,Sumaré,0.7893
5,6,Campinas,0.7876
6,7,Osasco,0.7868
7,8,Pedreira,0.7815
8,9,Santo André,0.7803
9,10,Sarutaiá,0.7789


# AHP-Claude

In [27]:
# JULGAMENTOS CLAUDE
# Exemplo: ("Dist_230kV", "FEC", "FEC", 3)
# Significa que FEC é 3 vezes mais importante.
# Nota 1 significa igual importância.

julgamentos_claude = [
    ("Dist_230kV", "FEC", "Dist_230kV", 2),
    ("Dist_230kV", "Tarifa_A4", "Dist_230kV", 3),
    ("Dist_230kV", "Dist_min_IX", "Dist_230kV", 3),
    ("Dist_230kV", "ISH", "Dist_230kV", 6),
    ("Dist_230kV", "IFA1001", "Dist_230kV", 9),
    ("Dist_230kV", "T2M", "Dist_230kV", 4),
    ("Dist_230kV", "MDO_custo", "Dist_230kV", 9),
    ("Dist_230kV", "MDO_vinculos", "Dist_230kV", 9),
    ("Dist_230kV", "Roubo_furto_100k_hab", "Dist_230kV", 6),

    ("FEC", "Tarifa_A4", "FEC", 2),
    ("FEC", "Dist_min_IX", "FEC", 2),
    ("FEC", "ISH", "FEC", 4),
    ("FEC", "IFA1001", "FEC", 8),
    ("FEC", "T2M", "FEC", 3),
    ("FEC", "MDO_custo", "FEC", 9),
    ("FEC", "MDO_vinculos", "FEC", 6),
    ("FEC", "Roubo_furto_100k_hab", "FEC", 4),

    ("Tarifa_A4", "Dist_min_IX", "Tarifa_A4", 1),
    ("Tarifa_A4", "ISH", "Tarifa_A4", 2),
    ("Tarifa_A4", "IFA1001", "Tarifa_A4", 4),
    ("Tarifa_A4", "T2M", "Tarifa_A4", 2),
    ("Tarifa_A4", "MDO_custo", "Tarifa_A4", 6),
    ("Tarifa_A4", "MDO_vinculos", "Tarifa_A4", 3),
    ("Tarifa_A4", "Roubo_furto_100k_hab", "Tarifa_A4", 2),

    ("Dist_min_IX", "ISH", "Dist_min_IX", 2),
    ("Dist_min_IX", "IFA1001", "Dist_min_IX", 4),
    ("Dist_min_IX", "T2M", "Dist_min_IX", 2),
    ("Dist_min_IX", "MDO_custo", "Dist_min_IX", 6),
    ("Dist_min_IX", "MDO_vinculos", "Dist_min_IX", 3),
    ("Dist_min_IX", "Roubo_furto_100k_hab", "Dist_min_IX", 2),

    ("ISH", "IFA1001", "ISH", 2),
    ("ISH", "T2M", "ISH", 1),
    ("ISH", "MDO_custo", "ISH", 3),
    ("ISH", "MDO_vinculos", "ISH", 2),
    ("ISH", "Roubo_furto_100k_hab", "ISH", 1),

    ("IFA1001", "T2M", "T2M", 3),
    ("IFA1001", "MDO_custo", "IFA1001", 2),
    ("IFA1001", "MDO_vinculos", "IFA1001", 1),
    ("IFA1001", "Roubo_furto_100k_hab", "Roubo_furto_100k_hab", 2),

    ("T2M", "MDO_custo", "T2M", 4),
    ("T2M", "MDO_vinculos", "T2M", 2),
    ("T2M", "Roubo_furto_100k_hab", "T2M", 1),

    ("MDO_custo", "MDO_vinculos", "MDO_vinculos", 2),
    ("MDO_custo", "Roubo_furto_100k_hab", "Roubo_furto_100k_hab", 3),

    ("MDO_vinculos", "Roubo_furto_100k_hab", "Roubo_furto_100k_hab", 2)
]

print(f"Julgamentos Claude informados: {len(julgamentos_claude)}")  # deve imprimir 45

Julgamentos Claude informados: 45


In [28]:
n = len(criterios_ahp)

# Criar a matriz com diagonal igual a 1
matriz_claude = pd.DataFrame(
    np.eye(n, dtype=float),
    index=criterios_ahp,
    columns=criterios_ahp
)

# Preencher os julgamentos e seus valores recíprocos
for a, b, mais_importante, nota in julgamentos_claude:

    if mais_importante == a:
        valor_ab = float(nota)
    else:
        valor_ab = 1.0 / nota

    matriz_claude.loc[a, b] = valor_ab
    matriz_claude.loc[b, a] = 1.0 / valor_ab

# Validar diagonal e reciprocidade
assert np.allclose(np.diag(matriz_claude), 1.0)

assert np.allclose(
    matriz_claude.to_numpy() * matriz_claude.to_numpy().T,
    1.0
)

print(f"Matriz AHP-Claude criada: {n} x {n}")
display(matriz_claude.round(3))

Matriz AHP-Claude criada: 10 x 10


,Dist_230kV,FEC,Tarifa_A4,Dist_min_IX,ISH,IFA1001,T2M,MDO_custo,MDO_vinculos,Roubo_furto_100k_hab
Dist_230kV,1.000,2.000,3.000,3.000,6.000,9.0,4.000,9.0,9.0,6.000
FEC,0.500,1.000,2.000,2.000,4.000,8.0,3.000,9.0,6.0,4.000
Tarifa_A4,0.333,0.500,1.000,1.000,2.000,4.0,2.000,6.0,3.0,2.000
Dist_min_IX,0.333,0.500,1.000,1.000,2.000,4.0,2.000,6.0,3.0,2.000
ISH,0.167,0.250,0.500,0.500,1.000,2.0,1.000,3.0,2.0,1.000
IFA1001,0.111,0.125,0.250,0.250,0.500,1.0,0.333,2.0,1.0,0.500
T2M,0.250,0.333,0.500,0.500,1.000,3.0,1.000,4.0,2.0,1.000
MDO_custo,0.111,0.111,0.167,0.167,0.333,0.5,0.250,1.0,0.5,0.333
MDO_vinculos,0.111,0.167,0.333,0.333,0.500,1.0,0.500,2.0,1.0,0.500
Roubo_furto_100k_hab,0.167,0.250,0.500,0.500,1.000,2.0,1.000,3.0,2.0,1.000


In [29]:
# NORMALIZAÇÃO DA MATRIZ AHP-Claude
# Dividir cada elemento pela soma de sua coluna

matriz_claude_normalizada = matriz_claude.div(
    matriz_claude.sum(axis=0),
    axis=1
)

# CÁLCULO DOS PESOS
# Média de cada linha da matriz normalizada

pesos_claude = matriz_claude_normalizada.mean(axis=1)
pesos_claude.name = "Peso_Claude"

# Tabela equivalente à Tabela 3.3 do TCC

tabela_pesos_claude = pd.DataFrame({
    "Criterio": [nomes_ahp[col] for col in criterios_ahp],
    "Peso": pesos_claude.loc[criterios_ahp].to_numpy()
})

tabela_pesos_claude.loc[len(tabela_pesos_claude)] = [
    "Total",
    tabela_pesos_claude["Peso"].sum()
]

print("Matriz AHP-Claude normalizada:")
display(matriz_claude_normalizada.round(4))

print("Pesos dos critérios (Claude):")
display(tabela_pesos_claude.round(5))

Matriz AHP-Claude normalizada:


,Dist_230kV,FEC,Tarifa_A4,Dist_min_IX,ISH,IFA1001,T2M,MDO_custo,MDO_vinculos,Roubo_furto_100k_hab
Dist_230kV,0.3243,0.3820,0.3243,0.3243,0.3273,0.2609,0.2652,0.2000,0.3051,0.3273
FEC,0.1622,0.1910,0.2162,0.2162,0.2182,0.2319,0.1989,0.2000,0.2034,0.2182
Tarifa_A4,0.1081,0.0955,0.1081,0.1081,0.1091,0.1159,0.1326,0.1333,0.1017,0.1091
Dist_min_IX,0.1081,0.0955,0.1081,0.1081,0.1091,0.1159,0.1326,0.1333,0.1017,0.1091
ISH,0.0541,0.0477,0.0541,0.0541,0.0545,0.0580,0.0663,0.0667,0.0678,0.0545
IFA1001,0.0360,0.0239,0.0270,0.0270,0.0273,0.0290,0.0221,0.0444,0.0339,0.0273
T2M,0.0811,0.0637,0.0541,0.0541,0.0545,0.0870,0.0663,0.0889,0.0678,0.0545
MDO_custo,0.0360,0.0212,0.0180,0.0180,0.0182,0.0145,0.0166,0.0222,0.0169,0.0182
MDO_vinculos,0.0360,0.0318,0.0360,0.0360,0.0273,0.0290,0.0331,0.0444,0.0339,0.0273
Roubo_furto_100k_hab,0.0541,0.0477,0.0541,0.0541,0.0545,0.0580,0.0663,0.0667,0.0678,0.0545


Pesos dos critérios (Claude):


,Criterio,Peso
0,Disponibilidade de energia,0.30406
1,Confiabilidade de energia,0.20561
2,Custo de energia,0.11216
3,Latência,0.11216
4,Disponibilidade de água,0.05777
5,Custo da água,0.02979
6,Refrigeração,0.06719
7,Custo da mão de obra,0.01999
8,Disponibilidade de mão de obra,0.03350
9,Segurança física,0.05777


In [30]:
# Organizar os pesos na ordem de criterios_ranking

pesos_para_ranking_claude = pesos_claude.reindex(criterios_ranking)

# Validar a correspondência dos critérios
if pesos_para_ranking_claude.isna().any():
    raise ValueError("Existem critérios sem pesos AHP-Claude.")

# Validar a soma dos pesos
assert np.isclose(pesos_para_ranking_claude.sum(), 1.0)

print(f"Total de pesos: {len(pesos_para_ranking_claude)}")
print(f"Soma dos pesos: {pesos_para_ranking_claude.sum():.6f}")

print("Vetor de pesos Claude para SAW e TOPSIS:")
display(pesos_para_ranking_claude.round(5).to_frame("Peso_Claude"))

Total de pesos: 10
Soma dos pesos: 1.000000
Vetor de pesos Claude para SAW e TOPSIS:


,Peso_Claude
ISH,0.05777
MDO_vinculos,0.03350
Dist_230kV,0.30406
FEC,0.20561
Tarifa_A4,0.11216
Dist_min_IX,0.11216
IFA1001,0.02979
T2M,0.06719
MDO_custo,0.01999
Roubo_furto_100k_hab,0.05777


# SAW — AHP-Claude

In [31]:
# SAW - CÁLCULO DOS SCORES (AHP-Claude)

# Organizar pesos AHP-Claude na ordem da matriz normalizada
pesos_saw_claude = pesos_para_ranking_claude.reindex(
    matriz_normalizada.columns
)

# Multiplicar cada critério normalizado pelo seu peso AHP-Claude
matriz_ponderada_saw_claude = matriz_normalizada.mul(
    pesos_saw_claude,
    axis=1
)

# Aplicar a Equação 3.5: soma dos valores ponderados
scores_saw_claude = matriz_ponderada_saw_claude.sum(axis=1)

print("Scores SAW-Claude calculados:", len(scores_saw_claude))

display(
    scores_saw_claude.head().round(4).to_frame("Score_SAW_Claude")
)

Scores SAW-Claude calculados: 498


,Score_SAW_Claude
0,0.6873
1,0.6176
2,0.5910
3,0.7111
4,0.6428


In [32]:
# SAW - RANKING DOS MUNICÍPIOS (AHP-Claude)

# Criar base de resultados com identificação dos municípios
ranking_saw_claude = df_normalizacao[
    ["Codigo_municipio", "Municipio"]
].copy()

# Adicionar o score obtido pelo SAW
ranking_saw_claude["Score_SAW_Claude"] = scores_saw_claude

# Ordenar municípios pelo score, do maior para o menor
ranking_saw_claude = ranking_saw_claude.sort_values(
    by=["Score_SAW_Claude", "Municipio"],
    ascending=[False, True]
).reset_index(drop=True)

# Criar coluna de posição no ranking
ranking_saw_claude.insert(
    0,
    "Ranking_SAW_Claude",
    ranking_saw_claude.index + 1
)

# Apresentar os dez primeiros municípios
print("Ranking AHP-Claude-SAW - Top 10 municípios")

display(
    ranking_saw_claude[
        ["Ranking_SAW_Claude", "Municipio", "Score_SAW_Claude"]
    ].head(10).round(4)
)

Ranking AHP-Claude-SAW - Top 10 municípios


,Ranking_SAW_Claude,Municipio,Score_SAW_Claude
0,1,São Caetano do Sul,0.8530
1,2,Sumaré,0.8114
2,3,Diadema,0.8094
3,4,Pedreira,0.8068
4,5,Osasco,0.8034
5,6,Piraju,0.7993
6,7,Barueri,0.7966
7,8,Carapicuíba,0.7947
8,9,Campinas,0.7945
9,10,Monte Mor,0.7938


# TOPSIS — AHP-Claude

In [33]:
# TOPSIS - MATRIZ PONDERADA (AHP-Claude)


# Organizar os pesos AHP-Claude na ordem da matriz normalizada
pesos_topsis_claude = pesos_claude.reindex(
    matriz_normalizada.columns
)


# Multiplicar cada critério normalizado pelo respectivo peso
# Equação 3.6
matriz_ponderada_topsis_claude = matriz_normalizada.mul(
    pesos_topsis_claude,
    axis=1
)


print("Municípios:", len(matriz_ponderada_topsis_claude))
print("Critérios:", len(matriz_ponderada_topsis_claude.columns))


display(matriz_ponderada_topsis_claude.head().round(4))

Municípios: 498
Critérios: 10


,ISH,MDO_vinculos,Dist_230kV,FEC,Tarifa_A4,Dist_min_IX,IFA1001,T2M,MDO_custo,Roubo_furto_100k_hab
0,0.0578,0.0123,0.2680,0.1760,0.0140,0.0580,0.0264,0.0101,0.0164,0.0483
1,0.0385,0.0000,0.1602,0.1383,0.1004,0.0949,0.0268,0.0097,0.0000,0.0487
2,0.0193,0.0078,0.1849,0.1666,0.0126,0.0868,0.0268,0.0371,0.0155,0.0338
3,0.0385,0.0030,0.2612,0.1861,0.0126,0.0810,0.0264,0.0371,0.0137,0.0515
4,0.0193,0.0066,0.1646,0.1686,0.0520,0.0946,0.0278,0.0393,0.0191,0.0510


In [34]:
# TOPSIS - SOLUÇÕES IDEAIS (AHP-Claude)

# Solução Ideal Positiva - Equação 3.7
ideal_positivo_claude = matriz_ponderada_topsis_claude.max(axis=0)


# Solução Ideal Negativa - Equação 3.8
ideal_negativo_claude = matriz_ponderada_topsis_claude.min(axis=0)


# Organizar os resultados
tabela_ideais_claude = pd.DataFrame({
    "Ideal_Positivo": ideal_positivo_claude,
    "Ideal_Negativo": ideal_negativo_claude
})


print("Soluções ideais por critério (Claude):")
display(tabela_ideais_claude.round(4))

Soluções ideais por critério (Claude):


,Ideal_Positivo,Ideal_Negativo
ISH,0.0578,0.0
MDO_vinculos,0.0335,0.0
Dist_230kV,0.3041,0.0
FEC,0.2056,0.0
Tarifa_A4,0.1122,0.0
Dist_min_IX,0.1122,0.0
IFA1001,0.0298,0.0
T2M,0.0672,0.0
MDO_custo,0.0200,0.0
Roubo_furto_100k_hab,0.0578,0.0


In [35]:
# TOPSIS - DISTÂNCIAS EUCLIDIANAS (AHP-Claude)


# Distância até a Solução Ideal Positiva - Equação 3.9
distancia_positiva_claude = np.sqrt(
    (
        matriz_ponderada_topsis_claude.sub(
            ideal_positivo_claude, axis=1
        ) ** 2
    ).sum(axis=1)
)


# Distância até a Solução Ideal Negativa - Equação 3.10
distancia_negativa_claude = np.sqrt(
    (
        matriz_ponderada_topsis_claude.sub(
            ideal_negativo_claude, axis=1
        ) ** 2
    ).sum(axis=1)
)


# Consolidar as distâncias
tabela_distancias_claude = pd.DataFrame({
    "Distancia_Positiva": distancia_positiva_claude,
    "Distancia_Negativa": distancia_negativa_claude
})


print("Distâncias calculadas (Claude):", len(tabela_distancias_claude))


display(tabela_distancias_claude.head().round(4))

Distâncias calculadas (Claude): 498


,Distancia_Positiva,Distancia_Negativa
0,0.1362,0.3365
1,0.1759,0.2618
2,0.1730,0.2712
3,0.1241,0.3406
4,0.1667,0.2699


In [36]:
# TOPSIS - SCORES E RANKING FINAL (AHP-Claude)


# Calcular o coeficiente de proximidade - Equação 3.11
scores_topsis_claude = (
    distancia_negativa_claude
    / (distancia_negativa_claude + distancia_positiva_claude)
)


# Criar a base de resultados
ranking_topsis_claude = df_normalizacao[
    ["Codigo_municipio", "Municipio"]
].copy()


# Adicionar o score TOPSIS
ranking_topsis_claude["Score_TOPSIS_Claude"] = scores_topsis_claude


# Ordenar do maior para o menor score
ranking_topsis_claude = ranking_topsis_claude.sort_values(
    by=["Score_TOPSIS_Claude", "Municipio"],
    ascending=[False, True]
).reset_index(drop=True)


# Criar a posição no ranking
ranking_topsis_claude.insert(
    0,
    "Ranking_TOPSIS_Claude",
    ranking_topsis_claude.index + 1
)


# Apresentar os dez melhores municípios
print("Ranking AHP-Claude-TOPSIS - Top 10 municípios")


display(
    ranking_topsis_claude[
        ["Ranking_TOPSIS_Claude", "Municipio", "Score_TOPSIS_Claude"]
    ].head(10).round(4)
)

Ranking AHP-Claude-TOPSIS - Top 10 municípios


,Ranking_TOPSIS_Claude,Municipio,Score_TOPSIS_Claude
0,1,São Caetano do Sul,0.8338
1,2,Pedreira,0.8296
2,3,Sumaré,0.8292
3,4,Jaguariúna,0.8204
4,5,Paulínia,0.8178
5,6,Monte Mor,0.8173
6,7,Diadema,0.8160
7,8,Avaré,0.8136
8,9,Osasco,0.8133
9,10,Hortolândia,0.8110


# Comparação dos 4 modelos

Pesos lado a lado e uma tabela única com score e posição de cada município nos 4 modelos.

In [37]:
# PESOS AHP-GPT x AHP-CLAUDE

comparacao_pesos = pd.DataFrame({
    "Criterio": [nomes_ahp[c] for c in criterios_ahp],
    "Peso_GPT": pesos_gpt.loc[criterios_ahp].to_numpy(),
    "Peso_Claude": pesos_claude.loc[criterios_ahp].to_numpy()
})

comparacao_pesos["Diferenca"] = (
    comparacao_pesos["Peso_Claude"] - comparacao_pesos["Peso_GPT"]
)

display(comparacao_pesos.round(5))

,Criterio,Peso_GPT,Peso_Claude,Diferenca
0,Disponibilidade de energia,0.25579,0.30406,0.04827
1,Confiabilidade de energia,0.21139,0.20561,-0.00578
2,Custo de energia,0.11864,0.11216,-0.00648
3,Latência,0.08542,0.11216,0.02674
4,Disponibilidade de água,0.09375,0.05777,-0.03598
5,Custo da água,0.02556,0.02979,0.00424
6,Refrigeração,0.09921,0.06719,-0.03202
7,Custo da mão de obra,0.02556,0.01999,-0.00557
8,Disponibilidade de mão de obra,0.04235,0.03350,-0.00885
9,Segurança física,0.04235,0.05777,0.01543


In [38]:
# RANKINGS DOS 4 MODELOS EM UMA ÚNICA TABELA

chaves = ["Codigo_municipio", "Municipio"]

comparacao_rankings = (
    ranking_saw_gpt
    .merge(ranking_topsis_gpt, on=chaves)
    .merge(ranking_saw_claude, on=chaves)
    .merge(ranking_topsis_claude, on=chaves)
)

comparacao_rankings = comparacao_rankings[
    chaves + [
        "Ranking_SAW_GPT", "Score_SAW_GPT",
        "Ranking_TOPSIS_GPT", "Score_TOPSIS_GPT",
        "Ranking_SAW_Claude", "Score_SAW_Claude",
        "Ranking_TOPSIS_Claude", "Score_TOPSIS_Claude"
    ]
].sort_values("Ranking_SAW_GPT").reset_index(drop=True)

print("Municípios comparados:", len(comparacao_rankings))

display(comparacao_rankings.head(10).round(4))

Municípios comparados: 498


,Codigo_municipio,Municipio,Ranking_SAW_GPT,Score_SAW_GPT,Ranking_TOPSIS_GPT,Score_TOPSIS_GPT,Ranking_SAW_Claude,Score_SAW_Claude,Ranking_TOPSIS_Claude,Score_TOPSIS_Claude
0,3548807,São Caetano do Sul,1,0.8462,1,0.8166,1,0.8530,1,0.8338
1,3538808,Piraju,2,0.7958,2,0.7993,6,0.7993,13,0.8072
2,3513801,Diadema,3,0.7909,4,0.7899,3,0.8094,7,0.8160
3,3534401,Osasco,4,0.7848,7,0.7868,5,0.8034,9,0.8133
4,3552403,Sumaré,5,0.7806,5,0.7893,2,0.8114,3,0.8292
5,3509502,Campinas,6,0.7786,6,0.7876,9,0.7945,12,0.8096
6,3547809,Santo André,7,0.7733,9,0.7803,14,0.7887,17,0.8041
7,3504503,Avaré,8,0.7728,3,0.7961,16,0.7867,8,0.8136
8,3537107,Pedreira,9,0.7728,8,0.7815,4,0.8068,2,0.8296
9,3551207,Sarutaiá,10,0.7702,10,0.7789,29,0.7691,36,0.7781


In [39]:
# TOP 10 DE CADA MODELO LADO A LADO

top10_modelos = pd.DataFrame({
    "Posicao": range(1, 11),
    "AHP-GPT-SAW": ranking_saw_gpt["Municipio"].head(10).to_numpy(),
    "AHP-GPT-TOPSIS": ranking_topsis_gpt["Municipio"].head(10).to_numpy(),
    "AHP-Claude-SAW": ranking_saw_claude["Municipio"].head(10).to_numpy(),
    "AHP-Claude-TOPSIS": ranking_topsis_claude["Municipio"].head(10).to_numpy()
})

display(top10_modelos)

,Posicao,AHP-GPT-SAW,AHP-GPT-TOPSIS,AHP-Claude-SAW,AHP-Claude-TOPSIS
0,1,São Caetano do Sul,São Caetano do Sul,São Caetano do Sul,São Caetano do Sul
1,2,Piraju,Piraju,Sumaré,Pedreira
2,3,Diadema,Avaré,Diadema,Sumaré
3,4,Osasco,Diadema,Pedreira,Jaguariúna
4,5,Sumaré,Sumaré,Osasco,Paulínia
5,6,Campinas,Campinas,Piraju,Monte Mor
6,7,Santo André,Osasco,Barueri,Diadema
7,8,Avaré,Pedreira,Carapicuíba,Avaré
8,9,Pedreira,Santo André,Campinas,Osasco
9,10,Sarutaiá,Sarutaiá,Monte Mor,Hortolândia


In [40]:
# PESQUISA DE MUNICÍPIO NOS 4 MODELOS

import unicodedata

def _limpar(texto):
    # Ignora acentos, maiúsculas e espaços extras
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return texto.lower().strip()


def pesquisar_municipio(nome):

    busca = _limpar(nome)
    nomes = df["Municipio"].map(_limpar)

    # Primeiro tenta o nome exato; se não achar, busca por parte do nome
    encontrados = df[nomes == busca]
    if encontrados.empty:
        encontrados = df[nomes.str.contains(busca, regex=False)]

    if encontrados.empty:
        print(f"Nenhum município encontrado para '{nome}'.")
        return None

    total = len(comparacao_rankings)
    linhas = []

    for _, mun in encontrados.iterrows():

        if mun["Vetada"]:
            print(f"{mun['Municipio']} foi vetado no filtro inicial "
                  f"(motivo: {mun['Motivo_veto']}), por isso não entra nos rankings.")
            continue

        r = comparacao_rankings[
            comparacao_rankings["Codigo_municipio"] == mun["Codigo_municipio"]
        ].iloc[0]

        linhas.append({
            "Municipio": r["Municipio"],
            "AHP-GPT-SAW": int(r["Ranking_SAW_GPT"]),
            "AHP-GPT-TOPSIS": int(r["Ranking_TOPSIS_GPT"]),
            "AHP-Claude-SAW": int(r["Ranking_SAW_Claude"]),
            "AHP-Claude-TOPSIS": int(r["Ranking_TOPSIS_Claude"]),
        })

    if not linhas:
        return None

    tabela = pd.DataFrame(linhas)
    print(f"Posição no ranking (de {total} municípios elegíveis):")
    display(tabela.style.hide(axis="index"))
    return tabela


# ===== Digite aqui a cidade que quer pesquisar =====
pesquisar_municipio("Sao Paulo");

Posição no ranking (de 498 municípios elegíveis):


Municipio,AHP-GPT-SAW,AHP-GPT-TOPSIS,AHP-Claude-SAW,AHP-Claude-TOPSIS
São Paulo,22,28,27,30


In [41]:
# (OPCIONAL) EXPORTAR OS RESULTADOS PARA EXCEL

with pd.ExcelWriter("resultados_4_modelos.xlsx") as writer:
    comparacao_pesos.to_excel(writer, sheet_name="Pesos", index=False)
    comparacao_rankings.to_excel(writer, sheet_name="Rankings", index=False)
    top10_modelos.to_excel(writer, sheet_name="Top10", index=False)
    df[["Codigo_municipio", "Municipio", "Vetada", "Motivo_veto"]].to_excel(
        writer, sheet_name="Vetos", index=False
    )

files.download("resultados_4_modelos.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [42]:
# ============================================================
# MAPA INTERATIVO (HTML) — gera um arquivo .html com o mapa de SP
# Rode esta célula DEPOIS de todas as anteriores.
# ============================================================

import json, math, urllib.request
from datetime import datetime

URL_MALHA = "https://raw.githubusercontent.com/tbrugz/geodata-br/master/geojson/geojs-35-mun.json"


def _carregar_malha():
    # Malha municipal de SP (IBGE, via geodata-br). Baixa uma vez e reaproveita.
    global _MALHA_SP
    if "_MALHA_SP" in globals():
        return _MALHA_SP
    try:
        with urllib.request.urlopen(URL_MALHA, timeout=60) as r:
            _MALHA_SP = json.loads(r.read().decode("utf-8"))
    except Exception as erro:
        print("Não consegui baixar a malha automaticamente:", erro)
        print("Baixe o arquivo neste link e faça o upload a seguir:", URL_MALHA)
        up = files.upload()
        _MALHA_SP = json.loads(next(iter(up.values())).decode("utf-8"))
    return _MALHA_SP


def _compactar_malha(geo, largura=2000):
    # Projeta lon/lat em coordenadas de tela e grava cada município
    # como um caminho SVG compacto (deixa o HTML leve).
    lons, lats = [], []
    for f in geo["features"]:
        g = f["geometry"]
        polis = g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]
        for poli in polis:
            for anel in poli:
                for lon, lat in anel:
                    lons.append(lon); lats.append(lat)
    lon0, lon1, lat0, lat1 = min(lons), max(lons), min(lats), max(lats)
    kx = math.cos(math.radians((lat0 + lat1) / 2))
    esc = largura / ((lon1 - lon0) * kx)
    altura = round((lat1 - lat0) * esc)

    feats = []
    for f in geo["features"]:
        g = f["geometry"]
        polis = g["coordinates"] if g["type"] == "MultiPolygon" else [g["coordinates"]]
        partes = []
        for poli in polis:
            for anel in poli:
                pts = []
                for lon, lat in anel:
                    p = (round((lon - lon0) * kx * esc), round((lat1 - lat) * esc))
                    if not pts or p != pts[-1]:
                        pts.append(p)
                if len(pts) < 4:
                    continue
                d = f"M{pts[0][0]} {pts[0][1]}l" + " ".join(
                    f"{x - px} {y - py}" for (px, py), (x, y) in zip(pts, pts[1:])
                ) + "z"
                partes.append(d.replace(" -", "-"))
        feats.append([str(f["properties"]["id"]), "".join(partes), f["properties"]["name"]])
    return {"w": largura, "h": altura, "f": feats}


def gerar_mapa_html(modelo, metodo, pesos, arquivo_saida, aviso=None, baixar=True):
    """
    modelo: nome que aparece no título, ex. "AHP-GPT" ou "AHP-Claude"
    metodo: "SAW" ou "TOPSIS"
    pesos:  série de pesos do AHP (ex.: pesos_gpt ou pesos_claude)
    """
    geo = _carregar_malha()
    malha = _compactar_malha(geo)
    ids_malha = {f[0] for f in malha["f"]}
    por_6_digitos = {i[:6]: i for i in ids_malha}

    criterios = [
        {
            "col": c,
            "nome": nomes_ahp[c],
            "peso": float(pesos[c]),
            "direcao": "direto" if c in criterios_diretos else "inverso",
        }
        for c in criterios_ahp
    ]

    municipios, sem_malha = [], []
    for idx, linha in df.iterrows():
        cod = str(int(float(linha["Codigo_municipio"])))
        cod = cod if cod in ids_malha else por_6_digitos.get(cod[:6])
        if cod is None:
            sem_malha.append(linha["Municipio"])
            continue
        item = {"id": cod, "n": str(linha["Municipio"])}
        if bool(linha["Vetada"]):
            item["v"] = 1
            item["m"] = str(linha["Motivo_veto"])
        else:
            item["x"] = [round(float(matriz_normalizada.loc[idx, c]), 6) for c in criterios_ahp]
        municipios.append(item)

    dados = {
        "modelo": modelo,
        "metodo": metodo.upper(),
        "criterios": criterios,
        "municipios": municipios,
        "aviso": aviso,
        "gerado": datetime.now().strftime("%d/%m/%Y"),
    }

    cabeca, corpo = TEMPLATE_HTML.split("<!--CORPO-->")
    corpo = corpo.replace(
        "/*__DADOS__*/null", json.dumps(dados, ensure_ascii=False, separators=(",", ":"))
    ).replace(
        "/*__GEO__*/null", json.dumps(malha, ensure_ascii=False, separators=(",", ":"))
    )
    cabeca = cabeca.replace("__TITULO__", f"Mapa {modelo}-{metodo.upper()}")

    html = (
        '<!doctype html>\n<html lang="pt-BR">\n<head>\n<meta charset="utf-8">\n'
        '<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">\n'
        + cabeca + "\n</head>\n<body>\n" + corpo + "\n</body>\n</html>\n"
    )

    with open(arquivo_saida, "w", encoding="utf-8") as f:
        f.write(html)

    print(f"Arquivo gerado: {arquivo_saida} ({len(html.encode('utf-8')) / 1e6:.2f} MB)")
    print(f"Municípios no mapa: {len(municipios)} de {len(df)} da base")
    if sem_malha:
        print("Sem correspondência na malha do IBGE (confira o código):", sem_malha)

    if baixar:
        files.download(arquivo_saida)
    return arquivo_saida



# Modelo do HTML (não precisa mexer)
TEMPLATE_HTML = r'''<title>__TITULO__</title>
<meta name="claude:viewport-fit-sides" content="cover">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Instrument+Sans:wght@400;500;600&family=Schibsted+Grotesk:wght@600;700&family=IBM+Plex+Mono:wght@400;500&display=swap">
<style>
/* Layout: mapa grande à esquerda, painel de controle fixo à direita; Top 10 abaixo do mapa. Empilha no celular. */
:root {
  --bg: #f3f6f5;
  --surface: #ffffff;
  --fg: #16201c;
  --muted: #5a6762;
  --line: #d8e0dc;
  --accent: #1b5d8c;
  --accent-soft: #e2edf5;
  --ok: #1d7444;
  --ok-soft: #e1f2e7;
  --erro: #b3261e;
  --erro-soft: #fbe7e5;
  --vetado: #b7bfbb;
  --semdados: #e6ebe9;
  --borda-mapa: #ffffff;
  --sel: #10202a;
  --f-display: "Schibsted Grotesk", "Segoe UI", system-ui, sans-serif;
  --f-body: "Instrument Sans", "Segoe UI", system-ui, sans-serif;
  --f-mono: "IBM Plex Mono", ui-monospace, "SFMono-Regular", Menlo, monospace;
}
@media (prefers-color-scheme: dark) {
  :root:not([data-theme="light"]) {
    --bg: #0f1513; --surface: #171f1c; --fg: #e5ebe8; --muted: #98a6a0; --line: #2a3531;
    --accent: #7db6df; --accent-soft: #1b2d3a; --ok: #6fcf97; --ok-soft: #173326;
    --erro: #f28b82; --erro-soft: #3a1d1b; --vetado: #4b5451; --semdados: #222b28;
    --borda-mapa: #0f1513; --sel: #ffffff; color-scheme: dark;
  }
}
:root[data-theme="dark"] {
  --bg: #0f1513; --surface: #171f1c; --fg: #e5ebe8; --muted: #98a6a0; --line: #2a3531;
  --accent: #7db6df; --accent-soft: #1b2d3a; --ok: #6fcf97; --ok-soft: #173326;
  --erro: #f28b82; --erro-soft: #3a1d1b; --vetado: #4b5451; --semdados: #222b28;
  --borda-mapa: #0f1513; --sel: #ffffff; color-scheme: dark;
}

* { box-sizing: border-box; }
[hidden] { display: none !important; }
html, body { margin: 0; }
body {
  background: var(--bg);
  color: var(--fg);
  font: 15px/1.5 var(--f-body);
  padding-left: max(16px, env(safe-area-inset-left, 0px));
  padding-right: max(16px, env(safe-area-inset-right, 0px));
  padding-block: 24px 40px;
}
.pagina { max-width: 1320px; margin: 0 auto; display: grid; gap: 20px; }

/* Cabeçalho */
.topo { display: grid; gap: 6px; }
.voltar { justify-self: start; font: 500 13px/1 var(--f-body); color: var(--accent); text-decoration: none; padding: 6px 0; }
.voltar:hover { text-decoration: underline; }
.voltar:focus-visible { outline: 2px solid var(--accent); outline-offset: 2px; }
.eyebrow { font: 500 12px/1 var(--f-mono); letter-spacing: .08em; text-transform: uppercase; color: var(--muted); }
h1 { font: 700 clamp(26px, 3.4vw, 36px)/1.1 var(--f-display); margin: 0; text-wrap: balance; letter-spacing: -.01em; }
h1 .metodo { color: var(--accent); }
.sub { margin: 0; color: var(--muted); max-width: 70ch; }
.aviso { background: var(--accent-soft); color: var(--fg); border-radius: 8px; padding: 10px 14px; font-size: 14px; max-width: 80ch; }
h2 { font: 600 15px/1.3 var(--f-display); margin: 0; letter-spacing: .01em; }

/* Grade principal */
.grade { display: grid; gap: 20px; grid-template-columns: minmax(0, 1fr) 380px; grid-template-areas: "mapa painel" "ranking painel"; align-items: start; }
.mapa-bloco { grid-area: mapa; min-width: 0; }
.painel { grid-area: painel; display: grid; gap: 16px; min-width: 0; }
.ranking { grid-area: ranking; min-width: 0; }
@media (max-width: 980px) {
  .grade { grid-template-columns: minmax(0, 1fr); grid-template-areas: "mapa" "painel" "ranking"; }
}
.card { background: var(--surface); border: 1px solid var(--line); border-radius: 10px; padding: 16px; display: grid; gap: 12px; min-width: 0; }

/* Mapa */
.mapa-bloco { background: var(--surface); border: 1px solid var(--line); border-radius: 10px; overflow: hidden; display: grid; }
.mapa-barra { display: flex; flex-wrap: wrap; gap: 12px 20px; align-items: center; justify-content: space-between; padding: 12px 14px; border-bottom: 1px solid var(--line); }
.legenda { display: flex; flex-wrap: wrap; gap: 8px 16px; align-items: center; font-size: 13px; color: var(--muted); }
.escala { display: grid; gap: 3px; }
.escala-barra { width: 180px; max-width: 40vw; height: 10px; border-radius: 3px; background: linear-gradient(90deg, #c43c2c, #e8b64a 50%, #2e8b4f); }
.escala-num { display: flex; justify-content: space-between; font: 12px var(--f-mono); font-variant-numeric: tabular-nums; }
.chave { display: inline-flex; align-items: center; gap: 6px; }
.chave i { width: 12px; height: 12px; border-radius: 2px; display: inline-block; border: 1px solid var(--line); }
.zoom { display: flex; gap: 6px; }
.btn { font: 500 14px/1 var(--f-body); color: var(--fg); background: var(--surface); border: 1px solid var(--line); border-radius: 6px; padding: 8px 12px; cursor: pointer; }
.btn:hover { border-color: var(--accent); }
.btn:focus-visible, input:focus-visible { outline: 2px solid var(--accent); outline-offset: 2px; }
.btn-ic { width: 34px; padding: 8px 0; font-size: 16px; }
.btn-forte { background: var(--accent); color: var(--surface); border-color: var(--accent); }
.mapa-wrap { position: relative; touch-action: none; background: var(--surface); }
#mapa { display: block; width: 100%; height: auto; cursor: grab; user-select: none; }
#mapa.arrastando { cursor: grabbing; }
#mapa path { stroke: var(--borda-mapa); stroke-width: .6; vector-effect: non-scaling-stroke; stroke-linejoin: round; }
#mapa path.sel { stroke: var(--sel); stroke-width: 2.5; }
#mapa path:hover { stroke: var(--sel); stroke-width: 1.5; }
.tip { position: absolute; pointer-events: none; background: var(--fg); color: var(--bg); padding: 8px 10px; border-radius: 6px; font-size: 13px; line-height: 1.35; max-width: 240px; z-index: 2; box-shadow: 0 4px 14px rgb(0 0 0 / .18); }
.tip b { display: block; font-weight: 600; }
.tip span { font-family: var(--f-mono); font-variant-numeric: tabular-nums; }
.estado-pesos { display: flex; flex-wrap: wrap; gap: 8px; align-items: center; padding: 10px 14px; border-top: 1px solid var(--line); font-size: 13px; color: var(--muted); }
.pill { display: inline-flex; align-items: center; gap: 6px; font: 500 12px/1 var(--f-body); padding: 5px 9px; border-radius: 999px; background: var(--accent-soft); color: var(--accent); }
.pill.editado { background: var(--ok-soft); color: var(--ok); }

/* Busca */
.busca-linha { display: flex; gap: 8px; }
.campo { font: 15px var(--f-body); color: var(--fg); background: var(--bg); border: 1px solid var(--line); border-radius: 6px; padding: 8px 10px; min-width: 0; width: 100%; }
.rot-campo { font: 500 11px/1 var(--f-mono); letter-spacing: .06em; text-transform: uppercase; color: var(--muted); }
.resultado { display: grid; gap: 10px; }
.res-nome { font: 600 18px/1.2 var(--f-display); }
.res-pos { display: flex; align-items: baseline; gap: 8px; flex-wrap: wrap; }
.res-pos strong { font: 600 30px/1 var(--f-mono); font-variant-numeric: tabular-nums; }
.res-pos span { color: var(--muted); font-size: 14px; }
.res-meta { display: grid; grid-template-columns: auto 1fr; gap: 4px 12px; font-size: 14px; }
.res-meta dt { color: var(--muted); }
.res-meta dd { margin: 0; font-family: var(--f-mono); font-variant-numeric: tabular-nums; }
.barras { display: grid; gap: 6px; font-size: 13px; }
.barra-linha { display: grid; grid-template-columns: minmax(0, 1fr) 90px 44px; gap: 8px; align-items: center; }
.barra-linha .nome { overflow: hidden; text-overflow: ellipsis; white-space: nowrap; color: var(--muted); }
.barra-trilho { height: 8px; background: var(--semdados); border-radius: 4px; overflow: hidden; }
.barra-trilho i { display: block; height: 100%; background: var(--accent); }
.barra-linha .v { font: 12px var(--f-mono); text-align: right; font-variant-numeric: tabular-nums; }
.msg { font-size: 14px; color: var(--muted); }
.vetado-tag { display: inline-block; font: 500 12px/1 var(--f-body); padding: 5px 9px; border-radius: 999px; background: var(--semdados); color: var(--fg); }

/* Pesos */
.pesos-tab { width: 100%; border-collapse: collapse; font-size: 14px; }
.pesos-tab th { font: 500 11px/1.2 var(--f-mono); text-transform: uppercase; letter-spacing: .06em; color: var(--muted); text-align: left; padding: 0 0 6px; }
.pesos-tab th.num, .pesos-tab td.num { text-align: right; }
.pesos-tab td { padding: 5px 0; border-top: 1px solid var(--line); vertical-align: middle; }
.crit-nome { display: block; line-height: 1.25; }
.crit-col { display: block; font: 11px var(--f-mono); color: var(--muted); }
.orig { font: 13px var(--f-mono); color: var(--muted); font-variant-numeric: tabular-nums; padding-right: 10px !important; }
.peso-in { width: 70px; text-align: right; font: 14px var(--f-mono); font-variant-numeric: tabular-nums; padding: 6px 8px; }
.peso-in.invalido { border-color: var(--erro); background: var(--erro-soft); }
.soma { display: flex; justify-content: space-between; align-items: center; gap: 10px; padding: 10px 12px; border-radius: 8px; font-size: 14px; }
.soma.ok { background: var(--ok-soft); color: var(--ok); }
.soma.erro { background: var(--erro-soft); color: var(--erro); }
.soma strong { font: 600 16px var(--f-mono); font-variant-numeric: tabular-nums; }
.acoes { display: flex; flex-wrap: wrap; gap: 8px; }
.nota { font-size: 13px; color: var(--muted); margin: 0; }

/* Ranking */
.ranking .card { gap: 14px; }
.ranking-cab { display: flex; flex-wrap: wrap; justify-content: space-between; align-items: baseline; gap: 8px; }
.resumo { font-size: 14px; color: var(--muted); margin: 0; }
.tab-wrap { overflow-x: auto; }
.rank-tab { width: 100%; border-collapse: collapse; font-size: 14px; min-width: 480px; }
.rank-tab th { font: 500 11px/1.2 var(--f-mono); text-transform: uppercase; letter-spacing: .06em; color: var(--muted); text-align: left; padding: 0 8px 8px; border-bottom: 1px solid var(--line); }
.rank-tab td { padding: 8px; border-bottom: 1px solid var(--line); }
.rank-tab .num { text-align: right; font-family: var(--f-mono); font-variant-numeric: tabular-nums; }
.rank-tab tr { cursor: pointer; }
.rank-tab tbody tr:hover td { background: var(--accent-soft); }
.delta { display: inline-block; min-width: 46px; text-align: center; font: 500 12px/1 var(--f-mono); padding: 4px 6px; border-radius: 999px; }
.delta.sobe { background: var(--ok-soft); color: var(--ok); }
.delta.desce { background: var(--erro-soft); color: var(--erro); }
.delta.igual { color: var(--muted); }
.mov { display: grid; gap: 4px; font-size: 14px; }
.mov b { font-weight: 600; }

footer { color: var(--muted); font-size: 12px; max-width: 90ch; }
@media (prefers-reduced-motion: no-preference) {
  #mapa path { transition: fill .25s ease; }
}
</style>
<!--CORPO-->
<div class="pagina">
  <header class="topo">
    <a class="voltar" href="index.html">← Todos os mapas</a>
    <span class="eyebrow">Localização de data centers · Estado de São Paulo</span>
    <h1 id="titulo"></h1>
    <p class="sub" id="subtitulo"></p>
    <div class="aviso" id="aviso" hidden></div>
  </header>

  <main class="grade">
    <section class="mapa-bloco" aria-label="Mapa dos municípios">
      <div class="mapa-barra">
        <div class="legenda">
          <div class="escala">
            <div class="escala-barra" aria-hidden="true"></div>
            <div class="escala-num"><span id="esc-min">0</span><span id="esc-rotulo">Score</span><span id="esc-max">1</span></div>
          </div>
          <span class="chave"><i style="background: var(--vetado)"></i>Vetado</span>
          <span class="chave"><i style="background: var(--semdados)"></i>Sem dados</span>
        </div>
        <div class="zoom">
          <button class="btn btn-ic" id="z-mais" type="button" aria-label="Aproximar">+</button>
          <button class="btn btn-ic" id="z-menos" type="button" aria-label="Afastar">−</button>
          <button class="btn" id="z-reset" type="button">Ver estado inteiro</button>
        </div>
      </div>
      <div class="mapa-wrap" id="mapa-wrap">
        <svg id="mapa" role="img" aria-label="Mapa de calor dos municípios de São Paulo"></svg>
        <div class="tip" id="tip" hidden></div>
      </div>
      <div class="estado-pesos">
        <span>O mapa usa:</span>
        <span class="pill" id="pill-pesos">Pesos originais</span>
        <span id="estado-extra"></span>
      </div>
    </section>

    <aside class="painel">
      <section class="card" aria-labelledby="h-busca">
        <h2 id="h-busca">Buscar município</h2>
        <span class="rot-campo">Por nome</span>
        <form class="busca-linha" id="form-busca" autocomplete="off">
          <input class="campo" id="campo-busca" list="lista-mun" placeholder="Ex.: Campinas" aria-label="Nome do município">
          <datalist id="lista-mun"></datalist>
          <button class="btn btn-forte" type="submit">Buscar</button>
        </form>
        <span class="rot-campo">Por posição no ranking</span>
        <form class="busca-linha" id="form-pos" autocomplete="off" novalidate>
          <input class="campo" id="campo-pos" type="number" min="1" step="1" inputmode="numeric" aria-label="Posição no ranking">
          <button class="btn btn-forte" type="submit">Ir</button>
        </form>
        <div class="resultado" id="resultado"><p class="msg">Digite o nome de uma cidade ou clique no mapa.</p></div>
      </section>

      <section class="card" aria-labelledby="h-pesos">
        <h2 id="h-pesos">Pesos dos critérios</h2>
        <table class="pesos-tab">
          <thead><tr><th>Critério</th><th class="num">Original</th><th class="num">Peso</th></tr></thead>
          <tbody id="pesos-corpo"></tbody>
        </table>
        <div class="soma" id="soma" role="status" aria-live="polite"></div>
        <div class="acoes">
          <button class="btn" id="btn-restaurar" type="button">Restaurar pesos originais</button>
          <button class="btn" id="btn-iguais" type="button">Pesos iguais (0,10)</button>
        </div>
        <p class="nota">Use até duas casas decimais (vírgula ou ponto). O mapa só atualiza quando a soma for exatamente 1,00.</p>
      </section>
    </aside>

    <section class="ranking" aria-labelledby="h-top">
      <div class="card">
        <div class="ranking-cab">
          <h2 id="h-top">Top 10</h2>
          <p class="resumo" id="resumo"></p>
        </div>
        <div class="tab-wrap">
          <table class="rank-tab">
            <thead><tr><th class="num">Posição</th><th>Município</th><th class="num">Score</th><th class="num">Posição original</th><th class="num">Variação</th></tr></thead>
            <tbody id="top-corpo"></tbody>
          </table>
        </div>
        <div class="mov" id="mov"></div>
      </div>
    </section>
  </main>

  <footer id="rodape"></footer>
</div>

<script>
const DADOS = /*__DADOS__*/null;
const GEO = /*__GEO__*/null;

(function () {
  "use strict";
  const $ = (id) => document.getElementById(id);
  const fmt = (v, c) => v.toLocaleString("pt-BR", { minimumFractionDigits: c, maximumFractionDigits: c });
  const semAcento = (s) => String(s).normalize("NFD").replace(/[̀-ͯ]/g, "").toLowerCase().trim();

  const crit = DADOS.criterios;
  const K = crit.length;
  const metodo = DADOS.metodo.toUpperCase();
  const pesosOrig = crit.map((c) => c.peso);

  // ---------- Municípios: base + malha ----------
  const muns = DADOS.municipios.map((m) => ({ id: m.id, n: m.n, v: !!m.v, m: m.m || "", x: m.x }));
  const porId = new Map(muns.map((m, i) => [m.id, i]));
  GEO.f.forEach(([id, , nome]) => {
    if (!porId.has(id)) { porId.set(id, muns.length); muns.push({ id, n: nome, v: false, m: "", x: null, semDados: true }); }
  });
  muns.forEach((m) => { if (!m.x && !m.v) m.semDados = true; });
  const eleg = [];
  muns.forEach((m, i) => { if (!m.v && m.x) eleg.push(i); });

  // ---------- Cálculo SAW / TOPSIS ----------
  function calcular(w) {
    const s = new Float64Array(muns.length).fill(NaN);
    if (metodo === "TOPSIS") {
      const mx = new Array(K).fill(-Infinity), mn = new Array(K).fill(Infinity);
      for (const i of eleg) for (let j = 0; j < K; j++) {
        const v = w[j] * muns[i].x[j];
        if (v > mx[j]) mx[j] = v;
        if (v < mn[j]) mn[j] = v;
      }
      for (const i of eleg) {
        let dp = 0, dn = 0;
        for (let j = 0; j < K; j++) {
          const v = w[j] * muns[i].x[j];
          dp += (v - mx[j]) ** 2; dn += (v - mn[j]) ** 2;
        }
        dp = Math.sqrt(dp); dn = Math.sqrt(dn);
        s[i] = dp + dn === 0 ? 0 : dn / (dn + dp);
      }
    } else {
      for (const i of eleg) {
        let t = 0;
        for (let j = 0; j < K; j++) t += w[j] * muns[i].x[j];
        s[i] = t;
      }
    }
    const ordem = eleg.slice().sort((a, b) => (s[b] - s[a]) || (muns[a].n < muns[b].n ? -1 : muns[a].n > muns[b].n ? 1 : 0));
    const pos = new Int32Array(muns.length);
    ordem.forEach((i, k) => { pos[i] = k + 1; });
    let min = Infinity, max = -Infinity;
    for (const i of eleg) { if (s[i] < min) min = s[i]; if (s[i] > max) max = s[i]; }
    return { s, ordem, pos, min, max };
  }

  const base = calcular(pesosOrig);
  let atual = base;
  let usandoOriginais = true;

  // ---------- Cores ----------
  const stops = [[196, 60, 44], [232, 182, 74], [46, 139, 79]];
  function cor(t) {
    t = Math.max(0, Math.min(1, t));
    const k = t < 0.5 ? 0 : 1, u = t < 0.5 ? t * 2 : (t - 0.5) * 2;
    const a = stops[k], b = stops[k + 1];
    return `rgb(${Math.round(a[0] + (b[0] - a[0]) * u)},${Math.round(a[1] + (b[1] - a[1]) * u)},${Math.round(a[2] + (b[2] - a[2]) * u)})`;
  }

  // ---------- Mapa ----------
  const svg = $("mapa");
  const NS = "http://www.w3.org/2000/svg";
  const W = GEO.w, H = GEO.h;
  let vb = { x: 0, y: 0, w: W, h: H };
  const setVB = () => svg.setAttribute("viewBox", `${vb.x} ${vb.y} ${vb.w} ${vb.h}`);
  setVB();
  const g = document.createElementNS(NS, "g");
  svg.appendChild(g);
  const paths = new Map();
  GEO.f.forEach(([id, d]) => {
    const p = document.createElementNS(NS, "path");
    p.setAttribute("d", d);
    p.dataset.i = porId.get(id);
    g.appendChild(p);
    paths.set(porId.get(id), p);
  });

  function pintar() {
    const r = atual, amp = r.max - r.min || 1;
    paths.forEach((p, i) => {
      const m = muns[i];
      if (m.v) p.style.fill = "var(--vetado)";
      else if (m.semDados) p.style.fill = "var(--semdados)";
      else p.style.fill = cor((r.s[i] - r.min) / amp);
    });
    $("esc-min").textContent = fmt(r.min, 3);
    $("esc-max").textContent = fmt(r.max, 3);
  }

  // Tooltip
  const tip = $("tip"), wrap = $("mapa-wrap");
  function textoStatus(i) {
    const m = muns[i];
    if (m.v) return `Vetado · ${m.m}`;
    if (m.semDados) return "Sem dados na base";
    return `<span>${atual.pos[i]}º</span> de ${eleg.length} · score <span>${fmt(atual.s[i], 4)}</span>`;
  }
  svg.addEventListener("pointermove", (e) => {
    const p = e.target.closest("path");
    if (!p || arrasto.ativo) { tip.hidden = true; return; }
    const i = +p.dataset.i;
    tip.innerHTML = `<b>${esc(muns[i].n)}</b>${textoStatus(i)}`;
    tip.hidden = false;
    const r = wrap.getBoundingClientRect();
    let x = e.clientX - r.left + 14, y = e.clientY - r.top + 14;
    if (x + tip.offsetWidth > r.width) x = e.clientX - r.left - tip.offsetWidth - 14;
    if (y + tip.offsetHeight > r.height) y = e.clientY - r.top - tip.offsetHeight - 14;
    tip.style.left = x + "px"; tip.style.top = y + "px";
  });
  svg.addEventListener("pointerleave", () => { tip.hidden = true; });

  // Zoom e arrasto
  function zoomEm(fator, cx, cy) {
    const nw = Math.min(W, Math.max(W / 14, vb.w / fator));
    const f = nw / vb.w;
    vb = { x: cx - (cx - vb.x) * f, y: cy - (cy - vb.y) * f, w: nw, h: vb.h * f };
    limitar(); setVB();
  }
  function limitar() {
    vb.x = Math.min(Math.max(vb.x, -vb.w * 0.3), W - vb.w * 0.7);
    vb.y = Math.min(Math.max(vb.y, -vb.h * 0.3), H - vb.h * 0.7);
  }
  function ptSVG(e) {
    const r = svg.getBoundingClientRect();
    return { x: vb.x + (e.clientX - r.left) / r.width * vb.w, y: vb.y + (e.clientY - r.top) / r.height * vb.h };
  }
  svg.addEventListener("wheel", (e) => {
    e.preventDefault();
    const p = ptSVG(e);
    zoomEm(e.deltaY < 0 ? 1.25 : 0.8, p.x, p.y);
  }, { passive: false });
  const arrasto = { ativo: false, moveu: false, x: 0, y: 0, vx: 0, vy: 0 };
  svg.addEventListener("pointerdown", (e) => {
    Object.assign(arrasto, { ativo: true, moveu: false, x: e.clientX, y: e.clientY, vx: vb.x, vy: vb.y });
  });
  window.addEventListener("pointermove", (e) => {
    if (!arrasto.ativo) return;
    const dx = e.clientX - arrasto.x, dy = e.clientY - arrasto.y;
    if (!arrasto.moveu && Math.hypot(dx, dy) < 5) return;
    arrasto.moveu = true; svg.classList.add("arrastando"); tip.hidden = true;
    const r = svg.getBoundingClientRect();
    vb.x = arrasto.vx - dx / r.width * vb.w; vb.y = arrasto.vy - dy / r.height * vb.h;
    limitar(); setVB();
  });
  window.addEventListener("pointerup", (e) => {
    if (!arrasto.ativo) return;
    arrasto.ativo = false; svg.classList.remove("arrastando");
    if (!arrasto.moveu) {
      const p = e.target.closest && e.target.closest("#mapa path");
      if (p) selecionar(+p.dataset.i, false);
    }
  });
  $("z-mais").onclick = () => zoomEm(1.5, vb.x + vb.w / 2, vb.y + vb.h / 2);
  $("z-menos").onclick = () => zoomEm(1 / 1.5, vb.x + vb.w / 2, vb.y + vb.h / 2);
  $("z-reset").onclick = () => { vb = { x: 0, y: 0, w: W, h: H }; setVB(); };

  // ---------- Seleção / busca ----------
  let selecionado = null;
  const esc = (s) => String(s).replace(/[&<>"]/g, (c) => ({ "&": "&amp;", "<": "&lt;", ">": "&gt;", '"': "&quot;" }[c]));

  function selecionar(i, centralizar) {
    if (selecionado !== null && paths.has(selecionado)) paths.get(selecionado).classList.remove("sel");
    selecionado = i;
    const p = paths.get(i);
    if (p) {
      p.classList.add("sel");
      g.appendChild(p);
      if (centralizar) {
        const b = p.getBBox();
        const nw = Math.max(W / 6, b.width * 4), nh = nw * H / W;
        vb = { x: b.x + b.width / 2 - nw / 2, y: b.y + b.height / 2 - nh / 2, w: nw, h: nh };
        limitar(); setVB();
      }
    }
    mostrarResultado();
  }

  function deltaHTML(orig, novo) {
    const d = orig - novo;
    if (d > 0) return `<span class="delta sobe" title="Subiu ${d} posições">▲ ${d}</span>`;
    if (d < 0) return `<span class="delta desce" title="Desceu ${-d} posições">▼ ${-d}</span>`;
    return `<span class="delta igual" title="Sem mudança">=</span>`;
  }

  function mostrarResultado() {
    const out = $("resultado");
    if (selecionado === null) return;
    const i = selecionado, m = muns[i];
    let h = `<div class="res-nome">${esc(m.n)}</div>`;
    if (m.v) {
      h += `<div><span class="vetado-tag">Vetado</span></div><p class="msg">Motivo: ${esc(m.m)}. Municípios vetados não entram no ranking.</p>`;
    } else if (m.semDados) {
      h += `<p class="msg">Este município não está na base de dados.</p>`;
    } else {
      const pa = atual.pos[i], po = base.pos[i];
      h += `<div class="res-pos"><strong>${pa}º</strong><span>de ${eleg.length} municípios elegíveis</span></div>
        <dl class="res-meta">
          <dt>Score ${metodo}</dt><dd>${fmt(atual.s[i], 4)}</dd>
          <dt>Posição original</dt><dd>${po}º ${usandoOriginais ? "" : deltaHTML(po, pa)}</dd>
        </dl>
        <div class="barras"><span class="msg">Desempenho normalizado (0 a 1)</span>` +
        crit.map((c, j) => `<div class="barra-linha"><span class="nome" title="${esc(c.nome)}">${esc(c.nome)}</span><span class="barra-trilho"><i style="width:${(m.x[j] * 100).toFixed(1)}%"></i></span><span class="v">${fmt(m.x[j], 2)}</span></div>`).join("") +
        `</div>`;
    }
    out.innerHTML = h;
  }

  const listaNomes = muns.map((m, i) => ({ i, n: m.n, k: semAcento(m.n) })).sort((a, b) => a.n.localeCompare(b.n, "pt-BR"));
  $("lista-mun").innerHTML = listaNomes.map((o) => `<option value="${esc(o.n)}"></option>`).join("");

  function buscar(txt) {
    const q = semAcento(txt);
    if (!q) return;
    const achado = listaNomes.find((o) => o.k === q) || listaNomes.find((o) => o.k.startsWith(q)) || listaNomes.find((o) => o.k.includes(q));
    if (!achado) { $("resultado").innerHTML = `<p class="msg">Nenhum município encontrado para "${esc(txt)}". Confira a grafia.</p>`; return; }
    selecionar(achado.i, true);
  }
  $("form-busca").addEventListener("submit", (e) => { e.preventDefault(); buscar($("campo-busca").value); });
  $("campo-busca").addEventListener("change", (e) => { buscar(e.target.value); });

  // Busca por posição: usa o ranking atual (com os pesos que estiverem nos campos)
  $("campo-pos").max = eleg.length;
  $("campo-pos").placeholder = `1 a ${eleg.length}`;
  $("form-pos").addEventListener("submit", (e) => {
    e.preventDefault();
    const p = Number($("campo-pos").value);
    if (!Number.isInteger(p) || p < 1 || p > eleg.length) {
      $("resultado").innerHTML = `<p class="msg">Informe uma posição de 1 a ${eleg.length}.</p>`;
      return;
    }
    selecionar(atual.ordem[p - 1], true);
  });

  // ---------- Pesos ----------
  function arredondar2(w) {
    // Arredonda para centésimos mantendo soma 1,00 (maiores restos)
    const brutos = w.map((v) => v * 100), base = brutos.map(Math.floor);
    let falta = 100 - base.reduce((a, b) => a + b, 0);
    brutos.map((v, j) => [v - base[j], j]).sort((a, b) => b[0] - a[0]).slice(0, falta).forEach(([, j]) => { base[j] += 1; });
    return base;
  }
  const centOrig = arredondar2(pesosOrig);

  const corpo = $("pesos-corpo");
  corpo.innerHTML = crit.map((c, j) => `<tr>
      <td><span class="crit-nome">${esc(c.nome)}</span><span class="crit-col">${esc(c.col)}</span></td>
      <td class="num orig">${fmt(c.peso, 4)}</td>
      <td class="num"><input class="campo peso-in" id="peso-${j}" inputmode="decimal" aria-label="Peso de ${esc(c.nome)}"></td>
    </tr>`).join("");
  const inputs = crit.map((_, j) => $("peso-" + j));

  function lerCentavos(txt) {
    const t = String(txt).trim().replace(",", ".");
    if (!/^(\d+(\.\d{0,2})?|\.\d{1,2})$/.test(t)) return null;
    const v = Math.round(parseFloat(t) * 100);
    return v >= 0 && v <= 100 ? v : null;
  }

  function preencher(cents) { inputs.forEach((el, j) => { el.value = fmt(cents[j] / 100, 2); el.classList.remove("invalido"); }); }

  function aoEditar() {
    const cents = inputs.map((el) => lerCentavos(el.value));
    inputs.forEach((el, j) => el.classList.toggle("invalido", cents[j] === null));
    const somaEl = $("soma");
    if (cents.some((c) => c === null)) {
      somaEl.className = "soma erro";
      somaEl.innerHTML = `<span>Há pesos inválidos. Use números de 0 a 1 com até duas casas.</span>`;
      $("estado-extra").textContent = "Mapa mantido na última combinação válida.";
      return;
    }
    const soma = cents.reduce((a, b) => a + b, 0);
    if (soma !== 100) {
      const dif = Math.abs(100 - soma) / 100;
      somaEl.className = "soma erro";
      somaEl.innerHTML = `<span>Soma <strong>${fmt(soma / 100, 2)}</strong></span><span>${soma < 100 ? "Faltam" : "Sobram"} ${fmt(dif, 2)}</span>`;
      $("estado-extra").textContent = "Mapa mantido na última combinação válida.";
      return;
    }
    somaEl.className = "soma ok";
    somaEl.innerHTML = `<span>Soma <strong>1,00</strong></span><span>Mapa atualizado</span>`;
    $("estado-extra").textContent = "";
    const iguaisOrig = cents.every((c, j) => c === centOrig[j]);
    aplicar(iguaisOrig ? null : cents.map((c) => c / 100));
  }

  function aplicar(w) {
    usandoOriginais = w === null;
    atual = usandoOriginais ? base : calcular(w);
    const pill = $("pill-pesos");
    pill.textContent = usandoOriginais ? `Pesos originais do ${DADOS.modelo}` : "Pesos editados";
    pill.classList.toggle("editado", !usandoOriginais);
    pintar(); desenharTop(); mostrarResultado();
  }

  inputs.forEach((el) => {
    el.addEventListener("input", aoEditar);
    el.addEventListener("blur", () => {
      const c = lerCentavos(el.value);
      if (c !== null) el.value = fmt(c / 100, 2);
    });
  });
  $("btn-restaurar").onclick = () => { preencher(centOrig); aoEditar(); };
  $("btn-iguais").onclick = () => {
    const c = new Array(K).fill(Math.floor(100 / K));
    const resto = 100 - c.reduce((a, b) => a + b, 0);
    for (let j = 0; j < resto; j++) c[j] += 1;
    preencher(c); aoEditar();
  };

  // ---------- Top 10 ----------
  function desenharTop() {
    const top = atual.ordem.slice(0, 10);
    $("top-corpo").innerHTML = top.map((i) => `<tr data-i="${i}">
        <td class="num">${atual.pos[i]}º</td>
        <td>${esc(muns[i].n)}</td>
        <td class="num">${fmt(atual.s[i], 4)}</td>
        <td class="num">${base.pos[i]}º</td>
        <td class="num">${deltaHTML(base.pos[i], atual.pos[i])}</td>
      </tr>`).join("");
    const mudaram = eleg.filter((i) => atual.pos[i] !== base.pos[i]).length;
    $("resumo").textContent = usandoOriginais
      ? `Ranking com os pesos originais do ${DADOS.modelo} · ${eleg.length} municípios elegíveis`
      : `${mudaram} de ${eleg.length} municípios mudaram de posição`;
    const topBase = new Set(base.ordem.slice(0, 10)), topAtual = new Set(top);
    const entraram = top.filter((i) => !topBase.has(i)).map((i) => muns[i].n);
    const sairam = base.ordem.slice(0, 10).filter((i) => !topAtual.has(i)).map((i) => `${muns[i].n} (agora ${atual.pos[i]}º)`);
    $("mov").innerHTML = usandoOriginais ? "" : (entraram.length
      ? `<div><b>Entraram no Top 10:</b> ${esc(entraram.join(", "))}</div><div><b>Saíram do Top 10:</b> ${esc(sairam.join(", "))}</div>`
      : `<div>O Top 10 tem os mesmos municípios do ranking original.</div>`);
  }
  $("top-corpo").addEventListener("click", (e) => {
    const tr = e.target.closest("tr");
    if (tr) selecionar(+tr.dataset.i, true);
  });

  // ---------- Textos ----------
  const nomeModelo = `${DADOS.modelo}-${metodo}`;
  $("titulo").innerHTML = `Mapa ${esc(DADOS.modelo)}-<span class="metodo">${esc(metodo)}</span>`;
  $("subtitulo").textContent = `Score ${metodo} de cada município elegível com os pesos do ${DADOS.modelo}. Altere os pesos ao lado para ver como o ranking muda.`;
  $("esc-rotulo").textContent = `Score ${metodo}`;
  if (DADOS.aviso) { $("aviso").textContent = DADOS.aviso; $("aviso").hidden = false; }
  const nVet = muns.filter((m) => m.v).length;
  $("rodape").textContent = `${nomeModelo} · ${eleg.length} municípios elegíveis, ${nVet} vetados. Critérios normalizados por min-max (0 a 1). Cores relativas: verde = maior score, vermelho = menor score entre os elegíveis. Malha municipal: IBGE (via geodata-br).${DADOS.gerado ? " Gerado em " + DADOS.gerado + "." : ""}`;
  document.title = `Mapa ${nomeModelo}`;

  preencher(centOrig);
  aoEditar();
})();
</script>
'''


# ===== Gera o mapa AHP-GPT-SAW =====
gerar_mapa_html("AHP-GPT", "SAW", pesos_gpt, "mapa_ahp_gpt_saw.html")
gerar_mapa_html("AHP-GPT", "TOPSIS", pesos_gpt, "mapa_ahp_gpt_topsis.html")
gerar_mapa_html("AHP-Claude", "SAW", pesos_claude, "mapa_ahp_claude_saw.html")
gerar_mapa_html("AHP-Claude", "TOPSIS", pesos_claude, "mapa_ahp_claude_topsis.html")

Arquivo gerado: mapa_ahp_gpt_saw.html (0.33 MB)
Municípios no mapa: 645 de 645 da base


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Arquivo gerado: mapa_ahp_gpt_topsis.html (0.33 MB)
Municípios no mapa: 645 de 645 da base


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (3 outputs left out)

Arquivo gerado: mapa_ahp_claude_topsis.html (0.33 MB)
Municípios no mapa: 645 de 645 da base


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'mapa_ahp_claude_topsis.html'